# MS MARCO RARS-v2.2 FP32 development gate

## tl;dr

This notebook runs one frozen seed-42 Stage-A experiment. It builds only `inner_train` and `inner_validation`, warm-starts from FP32 PCA, removes the query gate and int8 quantization, re-mines boundary pairs every epoch, and applies the registered stop rule.

This checked-in notebook has no claimed result until it is executed top-to-bottom on Colab. The historical outer validation, clean MS MARCO test, BEIR NQ test, and TREC DL 2019 are not evaluated here.

## Context & Methods

The exact method and stop rule are frozen in `protocols/rars_v2_2_boundary_loss_development_v1.json`. Both conditions are required on inner validation: at least `+0.01135` Recall@10 over Base and at least `+0.005` over directly computed FP32 PCA.

### Key assumptions

- Drive contains the same MS MARCO 1M embeddings, frozen M32 index, clean RARS-v1 sidecars, and PCA sidecars used by the prior notebook.
- The implementation commit below has been pushed to the configured GitHub repository.
- A T4/CUDA runtime and at least 4 GB of free local disk are available.
- MS MARCO qrels are sparse; every unjudged candidate used as a negative is interpreted as `unjudged-as-negative`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, shutil, subprocess, sys
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'faiss-gpu-cu12==1.12.0', 'pytest>=8,<9'], check=True)

CORE_COMMIT = 'bb9b106e69b9a453756fd800665f701614ce67b3'
REPO_URL = 'https://github.com/ravan-chuang/Embedding_Compression_for_RAG_Retrieval.git'
REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v2_2')
WORK = Path('/content') / f'rars-v2.2-{CORE_COMMIT[:12]}'
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
BUNDLES = WORK / 'bundles'
CANDIDATE_CACHE = WORK / 'candidate-cache'

DRIVE = Path('/content/drive/MyDrive/rag-pq-checkpoints')
CACHE = DRIVE / 'msmarco_basis_gate0_cache'
CLEAN = DRIVE / 'rars_clean_split_v1'
PCA = DRIVE / 'rars_pca_comparator_v1'
INDEX = DRIVE / 'msmarco_1m_pq_residual_gate3/frozen_ivfpq_m32_nlist512.index'
OUTPUT = DRIVE / 'rars-v2.2-fp32-msmarco' / CORE_COMMIT[:12]
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', CORE_COMMIT], check=True)
head = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
dirty = subprocess.check_output(
    ['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip()
assert head == CORE_COMMIT, (head, CORE_COMMIT)
assert not dirty, dirty

required = [
    CACHE / 'embeddings.fp16.memmap',
    CACHE / 'doc_ids.int64.memmap',
    CACHE / 'query_vectors.fp32.npy',
    CACHE / 'qrels_subset.json',
    INDEX,
    PCA / 'bases/pca_unweighted_rank16.float32.npy',
    PCA / 'sidecars/scales_pca_rank16.float32.npy',
    PCA / 'sidecars/codes_pca_rank16.int8.memmap',
    CLEAN / 'selected_config.json',
    CLEAN / 'bases/score_error_weighted_rank16.npy',
    CLEAN / 'sidecars/scales_score_error_weighted_rank16.float32.npy',
    CLEAN / 'sidecars/codes_score_error_weighted_rank16.int8.memmap',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, {'missing_artifacts': missing}
assert shutil.disk_usage('/content').free >= 4_000_000_000, 'Need 4 GB local disk'
print('Exact clean implementation commit:', head)

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_rars_v2_2_core.py',
    'tests/test_freeze_rars_v2_2_inner_bundles.py',
    'tests/test_build_msmarco_rars_v2_boundary_bundles.py',
    'tests/test_boundary_loss_sidecar.py',
], cwd=REPO, check=True)

## Data

Build fresh candidate/residual bundles on ephemeral local disk. `--inner-only` prevents creation or scoring of the burned outer bundle. No prior candidate cache or full-residual cache is reused.

In [ ]:
builder = [
    sys.executable, str(REPO / 'scripts/build_msmarco_rars_v2_boundary_bundles.py'),
    '--inner-only',
    '--embeddings', str(CACHE / 'embeddings.fp16.memmap'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--query-vectors', str(CACHE / 'query_vectors.fp32.npy'),
    '--index', str(INDEX),
    '--qrels', str(CACHE / 'qrels_subset.json'),
    '--train-split', str(REPO / 'splits/msmarco_rars_train_split.json'),
    '--validation-split', str(REPO / 'splits/msmarco_rars_validation_split.json'),
    '--cache-root', str(CANDIDATE_CACHE),
    '--pca-config', str(REPO / 'results/rars_pca_comparator/selected_pca_config.json'),
    '--pca-basis', str(PCA / 'bases/pca_unweighted_rank16.float32.npy'),
    '--pca-scales', str(PCA / 'sidecars/scales_pca_rank16.float32.npy'),
    '--pca-codes', str(PCA / 'sidecars/codes_pca_rank16.int8.memmap'),
    '--rars-config', str(CLEAN / 'selected_config.json'),
    '--rars-basis', str(CLEAN / 'bases/score_error_weighted_rank16.npy'),
    '--rars-scales', str(CLEAN / 'sidecars/scales_score_error_weighted_rank16.float32.npy'),
    '--rars-codes', str(CLEAN / 'sidecars/codes_score_error_weighted_rank16.int8.memmap'),
    '--output-root', str(BUNDLES),
    '--residual-batch-size', '20000',
]
subprocess.run(builder, check=True)
bundle_summary = json.loads((BUNDLES / 'bundle_build_summary.json').read_text())
assert bundle_summary['outer_validation_built'] is False
assert set(bundle_summary['roles']) == {'inner_train', 'inner_validation'}
print(json.dumps(bundle_summary, indent=2))

In [ ]:
subprocess.run([
    sys.executable, str(REPO / 'scripts/freeze_rars_v2_2_inner_bundles.py'),
    '--bundle-root', str(BUNDLES),
    '--query-vectors', str(CACHE / 'query_vectors.fp32.npy'),
    '--train-split', str(REPO / 'splits/msmarco_rars_train_split.json'),
    '--outer-validation-split', str(REPO / 'splits/msmarco_rars_validation_split.json'),
    '--clean-test-split', str(REPO / 'splits/msmarco_rars_test_split.json'),
    '--source-commit', CORE_COMMIT,
], check=True)
freeze_summary = json.loads((BUNDLES / 'v2_2_freeze_summary.json').read_text())
assert freeze_summary['status'] == 'INNER_BUNDLES_FROZEN'
assert freeze_summary['outer_validation_built_or_read_by_freezer'] is False
for role in ('inner_train', 'inner_validation'):
    manifest = json.loads((BUNDLES / role / 'v2_2_manifest.json').read_text())
    assert manifest['source_commit'] == CORE_COMMIT
    assert manifest['data_access']['outer_outcomes_used'] is False
    assert manifest['data_access']['closed_test_relevance_values_used'] is False
print(json.dumps(freeze_summary, indent=2))

## Results

Run the single registered seed-42 FP32 gate. Epoch 0 is the bounded PCA parameter warm-start and remains eligible for selection. A non-empty partial output directory is rejected; an existing complete run is reused only when the full fingerprint and every output hash match.

In [ ]:
run_dir = OUTPUT / 'seed42-fp32-stage-a'
trainer = [
    sys.executable, str(REPO / 'scripts/train_boundary_loss_sidecar_v2_2.py'),
    '--bundle-dir', str(BUNDLES / 'inner_train'),
    '--selection-bundle-dir', str(BUNDLES / 'inner_validation'),
    '--pca-basis', str(PCA / 'bases/pca_unweighted_rank16.float32.npy'),
    '--pca-config', str(REPO / 'results/rars_pca_comparator/selected_pca_config.json'),
    '--output-dir', str(run_dir),
    '--source-commit', CORE_COMMIT,
    '--rank', '16', '--top-b', '40', '--final-k', '10',
    '--epochs', '10', '--batch-size', '2048', '--score-batch-size', '256',
    '--max-negatives-per-positive', '8', '--promotion-mix', '0.8',
    '--minimum-margin', '0.0001', '--margin-multiplier', '1.0',
    '--learning-rate', '0.0001', '--weight-decay', '0.0001',
    '--correction-l2', '0.001', '--max-correction', '0.05',
    '--max-grad-norm', '5.0', '--seed', '42',
    '--minimum-gain-over-base', '0.01135',
    '--minimum-gain-over-pca', '0.005',
    '--device', 'cuda', '--reuse-complete',
]
subprocess.run(trainer, cwd=REPO, check=True)
summary = json.loads((run_dir / 'training_summary.json').read_text())
metrics = json.loads((run_dir / 'selection_metrics.json').read_text())
assert summary['source_commit'] == CORE_COMMIT
assert summary['quantization'] == 'none'
assert summary['query_gate_present'] is False
assert not (run_dir / 'document_scales.float32.npy').exists()
assert not (run_dir / 'document_codes.int8.npy').exists()
assert not (run_dir / 'query_gate_weight.float32.npy').exists()
print(json.dumps(metrics, indent=2))

In [ ]:
decision = metrics['decision']
if decision == 'STOP_RANK16_LEARNED_SIDECAR':
    print('STOP: do not run QAT, do not tune on the burned outer split, and do not add seeds.')
elif decision == 'GO_TO_THREE_SEED_FP32_REPLICATION':
    print('PROVISIONAL GO: freeze an unchanged three-seed FP32 replication next; QAT is still forbidden.')
else:
    raise AssertionError(f'Unknown registered decision: {decision}')

## Takeaways

Interpret only the executed `selection_metrics.json`. `STOP_RANK16_LEARNED_SIDECAR` ends this rank-16 learned-sidecar direction. `GO_TO_THREE_SEED_FP32_REPLICATION` authorizes an unchanged FP32 replication protocol, not QAT or an outer/test claim.

Do not edit the method, split, margin, Top-B, loss mix, or thresholds after reading this result. Any change is v2.3 and needs a new protocol ID.